Part A – Loading and configuring a pre-trained LLM

In [41]:
!pip install -q transformers torch sentencepiece

In [42]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Student Roll No: 24BAD041")
print("Libraries imported successfully!")

Student Roll No: 24BAD041
Libraries imported successfully!


In [43]:
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [44]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Clear previous model from memory
del model
gc.collect()
torch.cuda.empty_cache()

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

print("FLAN-T5-Large loaded successfully!")
print("Running on:", next(model.parameters()).device)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5-Large loaded successfully!
Running on: cuda:0


In [45]:
prompt = "What is Artificial Intelligence?"

inputs = tokenizer(prompt, return_tensors="pt")

print("Input Prompt:", prompt)
print("Tokenized Input:", inputs["input_ids"])

Input Prompt: What is Artificial Intelligence?
Tokenized Input: tensor([[  363,    19, 24714,  5869,  2825,  1433,    58,     1]])


In [46]:
# Move inputs to the same device as the model (GPU)
inputs = {k: v.to("cuda") for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated Response:")
print(response)

Generated Response:
artificial intelligence abraham luther king


Part B – Implementing chatbot functionality

In [47]:
def chatbot(user_query):

    prompt = f"""
    You are a knowledgeable computer science tutor.
    Explain the following technical concept accurately.
    Give a clear definition and one or two important details.

    Question: {user_query}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [48]:
print("Student Roll No: 24BAD041")

print("User: What is Machine Learning?")
print("Bot:", chatbot("What is Machine Learning?"))

print("\nUser: What is Deep Learning?")
print("Bot:", chatbot("What is Deep Learning?"))

print("\nUser: What is Python?")
print("Bot:", chatbot("What is Python?"))

Student Roll No: 24BAD041
User: What is Machine Learning?
Bot: Machine Learning (ML) is the process of learning from data.

User: What is Deep Learning?
Bot: Deep Learning (DL) is a type of machine learning in which a machine learns to recognize and classify objects based on their physical properties.

User: What is Python?
Bot: Python is a computer programming language.


Part C – Developing an interactive interface using Gradio

In [49]:
!pip install -q gradio
import gradio as gr

print("Gradio Version:", gr.__version__)
print("GPU Available:", torch.cuda.is_available())

Gradio Version: 6.30.0
GPU Available: True


In [54]:

import gradio as gr
import torch

def chat_interface(message, history):

    try:
        prompt = (
            "You are a helpful computer science tutor. "
            "Answer the user's question clearly and accurately.\n"
        )

        # Include previous conversation
        for item in history[-4:]:

            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")

                if isinstance(content, str):
                    prompt += f"{role}: {content}\n"

            elif isinstance(item, (list, tuple)):
                if len(item) == 2:
                    prompt += f"User: {item[0]}\n"
                    prompt += f"Assistant: {item[1]}\n"

        prompt += f"User: {message}\nAssistant:"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                num_beams=4,
                do_sample=False
            )

        response = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        return response

    except Exception as e:
        print("Chatbot Error:", str(e))
        return "Sorry, an error occurred. Please check the Colab output."


demo = gr.ChatInterface(
    fn=chat_interface,
    title="LLM Based AI Chatbot",
    description="Student Roll No: 24BAD041"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7dc157c4d136a5885b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Part D – Application testing and deployment

In [52]:
print("User: Explain Transformer architecture in deep learning.")
print("Bot:", chatbot(
    "Explain Transformer architecture in deep learning "
    "and describe its self-attention mechanism."
))

User: Explain Transformer architecture in deep learning.
Bot: Transformer architecture in deep learning is a type of self-attention mechanism used in machine learning .


In [53]:
questions = [
    "Define Artificial Intelligence in computer science.",
    "Explain Machine Learning as a branch of AI.",
    "Explain Transformer architecture in deep learning.",
    "Define Natural Language Processing in AI."
]

for question in questions:
    print("\nUser:", question)
    print("Bot:", chatbot(question))


User: Define Artificial Intelligence in computer science.
Bot: Artificial Intelligence ( AI ) is the study of computer programs that are able to learn from past experiences and apply them to solve real-world problems.

User: Explain Machine Learning as a branch of AI.
Bot: Machine Learning is a branch of Artificial Intelligence , or AI .

User: Explain Transformer architecture in deep learning.
Bot: Transformer architecture is a type of deep learning architecture that uses a set of transformers to train a model.

User: Define Natural Language Processing in AI.
Bot: Natural Language Processing ( NLP ) is a branch of artificial intelligence ( AI ) that deals with the processing of natural language .


In [56]:
test_queries = [
    "What is Artificial Intelligence?",
    "Explain Machine Learning.",
    "What is a Transformer model in deep learning?",
    "What is Natural Language Processing?",
    "What are the applications of AI?"
]

print("Student Roll No: 24BAD041")
print("LLM CHATBOT TESTING")

for query in test_queries:
    print("\nUser:", query)
    print("Bot:", chatbot(query))

Student Roll No: 24BAD041
LLM CHATBOT TESTING

User: What is Artificial Intelligence?
Bot: Artificial Intelligence (AI) is the process of developing computer programs that can perform a variety of tasks.

User: Explain Machine Learning.
Bot: Machine Learning (ML) is a branch of computer science that uses algorithms to learn from data.

User: What is a Transformer model in deep learning?
Bot: A transformer model in deep learning is a machine learning model that uses a combination of neural networks and machine learning techniques to train a deep learning model.

User: What is Natural Language Processing?
Bot: Natural Language Processing ( NLP ) is the process of analyzing and synthesizing human language.

User: What are the applications of AI?
Bot: Artificial Intelligence (AI) is a branch of computer science that uses machine learning to solve problems.


In [57]:
%%writefile app.py

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Student Roll No: 24BAD041")

model_name = "google/flan-t5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


def chatbot(message, history):

    prompt = (
        "You are a helpful computer science tutor. "
        "Answer clearly and accurately in simple English.\n"
    )

    for item in history[-4:]:
        if isinstance(item, dict):
            prompt += f"{item['role']}: {item['content']}\n"

    prompt += f"user: {message}\nassistant:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


demo = gr.ChatInterface(
    fn=chatbot,
    title="LLM Based AI Chatbot",
    description="Student Roll No: 24BAD041",
    type="messages"
)

demo.launch()

Writing app.py


In [58]:
!ls


app.py	sample_data


In [59]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
%%writefile requirements.txt
torch
transformers
sentencepiece
gradio

Writing requirements.txt


In [61]:
%%writefile README.md

# LLM Based AI Chatbot

**Student Roll No:** 24BAD041

## Aim
To develop an interactive chatbot using a pre-trained Transformer-based LLM.

## Technologies Used
- Python
- PyTorch
- Hugging Face Transformers
- FLAN-T5-Large
- Gradio
- Google Colab

## Features
- User prompt processing
- AI-generated responses
- Conversation history
- Interactive chatbot interface

## Execution
Install dependencies:

pip install -r requirements.txt

Run the application:

python app.py

Writing README.md


In [62]:
import os

print("PROJECT FILES")

for file in ["app.py", "requirements.txt", "README.md"]:
    if os.path.exists(file):
        print(file, "- Created successfully")
    else:
        print(file, "- Not found")

PROJECT FILES
app.py - Created successfully
requirements.txt - Created successfully
README.md - Created successfully


In [63]:
!zip -r LLM_Chatbot.zip app.py requirements.txt README.md

from google.colab import files
files.download("LLM_Chatbot.zip")

  adding: app.py (deflated 50%)
  adding: requirements.txt (stored 0%)
  adding: README.md (deflated 34%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>